# kigurumi 头壳尺寸计算器 · Colab 版

从 **演员身体数据 + 角色设定比例 + 假发留量** 反推头壳该做多大，并校验戴不戴得上。

**怎么用**：从上往下依次运行。cell 1、2 各跑一次装好引擎；之后 A / B / C 三节可以反复调参重跑。

| 节 | 干什么 |
|---|---|
| **A** | 上传角色正面立绘，自动量出 头身比 `N` 与 头宽/肩宽 `r`（可跳过，手填也行） |
| **B** | 填演员实测数据 → 输出净壳 / 含发 的 长宽高、比例、可戴性校验、重量估算 |
| **C** | 扫描图：这个演员到底能做到几头身、肩要垫多宽 |

---

### 两点必须先说清楚

**A 节是几何启发式，不是识别模型。** 它靠剪影的宽度剖面找头顶/下巴/颈/肩四条线。长发盖住脖子、抱手臂、大裙摆、动态姿势都会骗过它。合成标定图上 `r` 误差约 1%，`N` 误差约 4%——误差几乎全在下巴线上，因为剪影里下巴和脖子的交界本来就是模糊的。**每次都会输出标注图，请看一眼再用结果**，四条线都能手动覆盖。

**假发留量的定义**：从该侧「头壳基底材质的外表面」，沿该方向垂直量到发丝外缘的距离。发套底网、定位胶带、发丝蓬松量全算在这一个数里；壁厚不算，它在壳外皮以内，不会重复计算。所以正面看：`含发头宽 = 净壳宽 + 2 × 侧向留量`。

> 本 notebook 自带一份计算逻辑的副本，和命令行版 `kigurumi_head_calc.py` / `char_ratio_from_image.py` 是两份独立代码，改一边不会同步到另一边。

## Cell 1 · 计算引擎（运行一次）

In [ ]:
# ============ 头壳几何计算引擎 ============
import math, json
from dataclasses import dataclass
from typing import Optional, List, Dict, Any

# --- 人体测量学经验常数（未实测时的兜底估算）---
ACROMION_RATIO       = 0.818   # 肩峰高 / 身高
EYE_FROM_VERTEX      = 0.44    # 眼中心到头顶 / 头高
HEAD_DEPTH_PER_WIDTH = 1.25    # 真人头深 / 头宽

# --- 动漫风格头壳的经验区间（不是硬标准，可按你的数据改）---
ASPECT_MIN, ASPECT_MAX = 0.68, 1.00   # 净壳 宽/高
ASPECT_TARGET          = 0.82
EYE_FROM_TOP_MIN, EYE_FROM_TOP_MAX = 0.50, 0.65  # 角色眼位到壳顶 / 净壳高

MATERIAL_DENSITY = {"pla":1.24, "petg":1.27, "abs":1.04, "resin":1.15, "glassfiber":1.60}

# 未单独指定时，各方向假发留量 = 基准值 × 系数。
# 侧向取 1.0 只是「没有更好信息时按顶部同量处理」——侧向留量和顶部蓬松量
# 本是两个独立的物理量，能量到就直接填实测值。
WIG_FACTOR = {"top":1.0, "side":1.0, "front":0.6, "back":1.4}

# 未单独指定时，各方向海绵厚度 = foam × 系数。
# 顶部要做头台承重所以最厚，面部要让出视线所以最薄。
FOAM_FACTOR = {"top":1.5, "side":1.0, "face":0.4, "back":1.0}
# 海绵不会铺满整个内壁（面窗、颈口、机构位都不贴），估重时按这个覆盖率打折。
FOAM_COVERAGE = 0.65


def auto_shoulder_pad(bmi):
    """偏瘦的人需要更多填充才能撑出角色轮廓。"""
    if bmi < 18.5: return 1.5
    if bmi < 24.0: return 1.0
    return 0.5


def ellipsoid_area(a, b, c):
    p = 1.6075
    return 4*math.pi*(((a*b)**p + (a*c)**p + (b*c)**p)/3)**(1/p)


def compute(height, weight, shoulder_width, head_width, head_height,
            ratio_N, head_shoulder_r,
            head_depth=None, shoulder_height=None,
            depth_ratio=1.15, hair_in_ratio=True,
            wig=3.0, wig_top=None, wig_side=None, wig_front=None, wig_back=None,
            foam=2.0, foam_top=None, foam_side=None, foam_face=None, foam_back=None,
            foam_compress=0.30, foam_density=0.05,
            mount="chin", rim_drop=0.0, lift=0.0,
            side_clear=1.0, top_clear=1.5, face_clear=2.0, back_clear=1.5,
            wall=0.5, shoulder_pad=None, material="pla"):
    """全部长度 cm，重量 kg。返回一个 dict。"""
    notes, warnings, advice = [], [], []
    N = float(ratio_N)
    if N <= 1.0:
        raise ValueError("头身比必须大于 1（否则身体高度为零或负）")

    # --- 1. 补全缺失的人体数据 ---
    if head_depth is None:
        head_depth = head_width * HEAD_DEPTH_PER_WIDTH
        notes.append("头深未实测，按 头宽 × %.2f 估算 = %.1f cm" % (HEAD_DEPTH_PER_WIDTH, head_depth))
    if shoulder_height is None:
        shoulder_height = height * ACROMION_RATIO
        notes.append("肩峰高未实测，按 身高 × %.3f 估算 = %.1f cm" % (ACROMION_RATIO, shoulder_height))

    bmi = weight / (height/100.0)**2
    if shoulder_pad is None:
        shoulder_pad = auto_shoulder_pad(bmi)
        notes.append("肩部填充未指定，按 BMI %.1f 自动取单侧 %.1f cm" % (bmi, shoulder_pad))

    # --- 2. 假发各方向留量（自壳体外表面量起）---
    wt = wig_top   if wig_top   is not None else wig*WIG_FACTOR["top"]
    ws_= wig_side  if wig_side  is not None else wig*WIG_FACTOR["side"]
    wf = wig_front if wig_front is not None else wig*WIG_FACTOR["front"]
    wb = wig_back  if wig_back  is not None else wig*WIG_FACTOR["back"]

    # --- 3. 底沿高度与内部净空需求 ---
    chin_height = height - head_height
    if mount == "shoulder":
        rim_base, mount_desc = shoulder_height, "肩支撑（底沿在肩峰线）"
    elif mount == "chin":
        rim_base, mount_desc = chin_height, "颈支撑（底沿在演员下巴线）"
    elif mount == "custom":
        rim_base, mount_desc = chin_height - rim_drop, "自定义（底沿低于下巴 %.1f cm）" % rim_drop
    else:
        raise ValueError("mount 只能是 shoulder / chin / custom")
    if rim_base <= 0:
        raise ValueError("底沿高度算出来 <= 0，请检查身高 / 头高 / rim_drop")
    if rim_base > chin_height + 0.01:
        warnings.append("底沿(%.1f cm)高于演员下巴(%.1f cm)，壳体会卡住下颌，"
                        "请改用 chin 或 custom" % (rim_base, chin_height))

    h_mount       = rim_base + lift      # 穿鞋后底沿离地
    vertex        = height + lift        # 穿鞋后头顶离地
    rim_to_vertex = vertex - h_mount     # 与增高无关的定值

    # --- 海绵：标称厚度 → 戴上后的实占厚度 ---
    # 海绵和空气余量是两回事：海绵是实体、可压缩、负责夹住头；
    # 空气余量是风扇/走线/通风/进出头的空隙，不可压缩。
    if not (0.0 <= foam_compress < 1.0):
        raise ValueError("海绵压缩率必须在 [0, 1) 之间")
    f_top  = foam_top  if foam_top  is not None else foam*FOAM_FACTOR["top"]
    f_side = foam_side if foam_side is not None else foam*FOAM_FACTOR["side"]
    f_face = foam_face if foam_face is not None else foam*FOAM_FACTOR["face"]
    f_back = foam_back if foam_back is not None else foam*FOAM_FACTOR["back"]
    keep = 1.0 - foam_compress
    e_top, e_side, e_face, e_back = f_top*keep, f_side*keep, f_face*keep, f_back*keep

    # 可戴最小净壳 = 人头 + 海绵实占 + 空气余量 + 壁厚
    hs_min = rim_to_vertex + e_top + top_clear + wall
    ws_min = head_width + 2*(e_side + side_clear + wall)
    ds_min = head_depth + e_face + face_clear + e_back + back_clear + 2*wall

    # --- 4. 由角色比例反推目标尺寸 ---
    if hair_in_ratio:
        h_visual = h_mount/(N-1);  hs = h_visual - wt
    else:
        hs = (h_mount + wt)/(N-1); h_visual = hs + wt
    total_height = h_mount + h_visual

    shoulder_cos = shoulder_width + 2*shoulder_pad
    w_visual     = head_shoulder_r * shoulder_cos
    ws           = w_visual - 2*ws_
    ds           = ws * depth_ratio
    d_visual     = ds + wf + wb

    # --- 5. 可行性校验 ---
    ok_h, ok_w, ok_d = hs >= hs_min, ws >= ws_min, ds >= ds_min

    if hair_in_ratio:
        need_mount_h = (N-1)*(hs_min + wt); n_max = 1 + h_mount/(hs_min + wt)
    else:
        need_mount_h = (N-1)*hs_min - wt;   n_max = 1 + (h_mount + wt)/hs_min
    extra_lift = need_mount_h - h_mount

    if not ok_h:
        warnings.append("净壳高 %.1f cm < 可戴最小值 %.1f cm，差 %.1f cm："
                        "角色头身比对这位演员偏「写实」了" % (hs, hs_min, hs_min-hs))
        advice.append("把增高加到 %.1f cm（现 %.1f，需再加 %.1f），或把头身比降到 %.2f 以下"
                      % (lift+extra_lift, lift, extra_lift, n_max))
    if hs > 45:
        warnings.append("净壳高 %.1f cm 超过 45 cm，重量与颈部负荷会很吃力" % hs)

    pad_needed = ((ws_min + 2*ws_)/head_shoulder_r - shoulder_width)/2
    if not ok_w:
        warnings.append("净壳宽 %.1f cm < 可戴最小值 %.1f cm，差 %.1f cm" % (ws, ws_min, ws_min-ws))
        advice.append("单侧肩部填充加到 %.1f cm（现 %.1f），或减薄侧面假发/内衬"
                      % (pad_needed, shoulder_pad))
    if not ok_d:
        warnings.append("净壳深 %.1f cm < 可戴最小值 %.1f cm，差 %.1f cm" % (ds, ds_min, ds_min-ds))
        advice.append("把 深/宽 比从 %.2f 提到 %.2f 以上" % (depth_ratio, ds_min/ws))

    # --- 6. 形状合理性 ---
    aspect = ws/hs if hs > 0 else float("nan")
    if hs > 0 and not (ASPECT_MIN <= aspect <= ASPECT_MAX):
        warnings.append("净壳 宽/高 = %.2f，%s（动漫头壳常见 %.2f~%.2f）"
                        % (aspect, "偏窄长" if aspect < ASPECT_MIN else "偏宽扁",
                           ASPECT_MIN, ASPECT_MAX))
        pad_balance = ((ASPECT_TARGET*hs + 2*ws_)/head_shoulder_r - shoulder_width)/2
        advice.append("若想把 宽/高 拉回 %.2f：单侧肩部填充改为 %.1f cm"
                      "（正数=垫宽，负数=收窄肩线）" % (ASPECT_TARGET, pad_balance))
        hs_bal = ws/ASPECT_TARGET
        n_bal  = (1 + h_mount/(hs_bal+wt)) if hair_in_ratio else (1 + (h_mount+wt)/hs_bal)
        if n_bal > 1:
            advice.append("或保持肩宽不动，把头身比改成 %.2f（当前 %.2f）——头身比与 头宽/肩宽 "
                          "这两个设定对这具身体是互相冲突的，必须让一个" % (n_bal, N))

    # --- 7. 眼位 ---
    eye_floor     = vertex - EYE_FROM_VERTEX*head_height
    eye_from_rim  = eye_floor - h_mount
    eye_from_top  = (h_mount + hs) - eye_floor
    eye_ratio     = eye_from_top/hs if hs > 0 else float("nan")
    eye_ok        = EYE_FROM_TOP_MIN <= eye_ratio <= EYE_FROM_TOP_MAX
    if hs > 0 and not eye_ok:
        warnings.append("演员眼位在净壳高的 %.0f%% 处（自壳顶下量），角色眼位一般在 %.0f%%~%.0f%%，"
                        "视线可能对不上眼窗" % (eye_ratio*100, EYE_FROM_TOP_MIN*100, EYE_FROM_TOP_MAX*100))

    # --- 8. 重量估算 ---
    density   = MATERIAL_DENSITY.get(material, MATERIAL_DENSITY["pla"])
    area      = ellipsoid_area(ws/2, ds/2, hs/2)*0.92    # 扣除颈口开孔
    shell_mass= area*wall*density/1000.0
    # 海绵按标称厚度算重（压缩不减少材料，只是占位变薄）
    area_in   = ellipsoid_area(max(0.1, ws/2-wall), max(0.1, ds/2-wall),
                               max(0.1, hs/2-wall))*0.92
    foam_mean = (f_top + 2*f_side + f_face + f_back)/5.0
    foam_mass = area_in*foam_mean*FOAM_COVERAGE*foam_density/1000.0
    total_mass= shell_mass + foam_mass
    mass_limit= min(2.5, 0.035*weight)
    if total_mass > mass_limit:
        warnings.append("壳体 %.2f + 海绵 %.2f = %.2f kg，已超过建议上限 %.2f kg"
                        "（涂装+假发+风扇还要再加 0.4~1.2 kg）"
                        % (shell_mass, foam_mass, total_mass, mass_limit))
        if shell_mass > foam_mass:
            advice.append("壁厚从 %.1f mm 减到 %.1f mm 可把总重压到 %.2f kg，或改用更轻的材料"
                          % (wall*10, max(0.1, wall*10*(mass_limit-foam_mass)/max(0.01, shell_mass)),
                             mass_limit))
        else:
            advice.append("重量主要在海绵上（%.2f kg）：改用更薄或更低密度的衬垫，"
                          "或只在受力点贴海绵而不是满铺" % foam_mass)

    return dict(
        bmi=bmi, mount_desc=mount_desc, h_mount=h_mount, rim_to_vertex=rim_to_vertex,
        shoulder_height=shoulder_height, chin_height=chin_height,
        shoulder_pad=shoulder_pad, shoulder_cos=shoulder_cos, head_depth=head_depth,
        wig=dict(top=wt, side=ws_, front=wf, back=wb),
        foam=dict(nom_top=f_top, nom_side=f_side, nom_face=f_face, nom_back=f_back,
                  eff_top=e_top, eff_side=e_side, eff_face=e_face, eff_back=e_back,
                  compress=foam_compress),
        chain=dict(
            W=[("演员头宽", head_width), ("海绵实占 2×%.2f" % e_side, 2*e_side),
               ("空气余量 2×%.2f" % side_clear, 2*side_clear), ("壁厚 2×%.2f" % wall, 2*wall)],
            H=[("底沿到演员头顶", rim_to_vertex), ("海绵实占(顶)", e_top),
               ("空气余量(顶)", top_clear), ("壁厚", wall)],
            D=[("演员头深", head_depth), ("海绵实占 面%.2f+后%.2f" % (e_face, e_back), e_face+e_back),
               ("空气余量 面%.2f+后%.2f" % (face_clear, back_clear), face_clear+back_clear),
               ("壁厚 2×%.2f" % wall, 2*wall)]),
        net=dict(D=ds, W=ws, H=hs), vis=dict(D=d_visual, W=w_visual, H=h_visual),
        minimum=dict(D=ds_min, W=ws_min, H=hs_min),
        ok=dict(D=ok_d, W=ok_w, H=ok_h, all=ok_d and ok_w and ok_h),
        total_height=total_height, n_max=n_max, extra_lift=max(0.0, extra_lift),
        pad_needed=pad_needed, aspect=aspect,
        eye=dict(from_rim=eye_from_rim, from_top=eye_from_top, ratio=eye_ratio, ok=eye_ok),
        mass=dict(material=material, wall_mm=wall*10, area=area, shell_kg=shell_mass,
                  foam_kg=foam_mass, total_kg=total_mass, limit_kg=mass_limit),
        notes=notes, warnings=warnings, advice=advice,
    )

print("✅ 计算引擎就绪")

## Cell 2 · 图像分析引擎（运行一次）

In [ ]:
# ============ 立绘 → 剪影 → 四条线 ============
import numpy as np, matplotlib, matplotlib.pyplot as plt
from PIL import Image

def _pick_cjk_font():
    """Colab 默认没有中文字体，图里会变成豆腐块。找得到就用，找不到图上转英文。"""
    import matplotlib.font_manager as fm
    for name in ("Noto Sans CJK JP", "Noto Sans CJK SC", "Noto Serif CJK JP",
                 "WenQuanYi Zen Hei", "Source Han Sans SC", "PingFang SC", "SimHei"):
        try:
            path = fm.findfont(name, fallback_to_default=False)
            if path:
                matplotlib.rcParams["font.family"] = name
                matplotlib.rcParams["axes.unicode_minus"] = False
                return True
        except Exception:
            pass
    return False

HAS_CJK = _pick_cjk_font()
def L(zh, en):
    """图上的标签：有中文字体用中文，没有就退英文，绝不显示豆腐块。"""
    return zh if HAS_CJK else en


def load_rgba(src, max_dim=1000):
    """src 可以是本地路径或 http(s) URL。返回 (RGBA ndarray, 原图/工作图 缩放系数)。"""
    if isinstance(src, str) and src.startswith(("http://", "https://")):
        import urllib.request, io
        with urllib.request.urlopen(src) as r:
            im = Image.open(io.BytesIO(r.read()))
    else:
        im = Image.open(src)
    im = im.convert("RGBA")
    ow, oh = im.size
    scale = max(1.0, max(ow, oh)/float(max_dim))
    if scale > 1.0:
        im = im.resize((max(1, int(ow/scale)), max(1, int(oh/scale))), Image.LANCZOS)
    return np.asarray(im), scale


def build_mask(a, alpha_thr=128, bg_tol=30):
    """有透明通道就用 alpha；否则从四边泛洪，把与边框同色的连通区判为背景。"""
    alpha = a[..., 3]
    if int((alpha < 250).sum()) > alpha.size * 0.002:
        return alpha >= alpha_thr, "alpha 通道"

    rgb = a[..., :3].astype(np.int16)
    border = np.concatenate([rgb[0], rgb[-1], rgb[:, 0], rgb[:, -1]])
    q = (border // 16)
    key = q[:, 0]*256 + q[:, 1]*16 + q[:, 2]
    bgq = np.bincount(key).argmax()
    bg = np.array([(bgq//256)*16+8, ((bgq//16) % 16)*16+8, (bgq % 16)*16+8])
    close = (np.abs(rgb - bg).max(axis=2) <= bg_tol)

    try:
        from scipy import ndimage
        lab, _ = ndimage.label(close)
        edge = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
        edge = edge[edge != 0]
        mask = ~np.isin(lab, edge)
    except ImportError:
        mask = ~close        # 没有 scipy 就退化成纯颜色判据（会误伤同色的衣服）
    return mask, "背景泛洪（背景色 RGB≈%d,%d,%d）" % tuple(bg)


def analyse(mask, ov=None, shoulder_band=0.6):
    ov = ov or {}
    warn = []
    h, w = mask.shape
    any_row = mask.any(axis=1)
    if not any_row.any():
        raise ValueError("剪影是空的：抠图失败。把 bg_tol 调大，或换带透明背景的 PNG。")

    left  = np.argmax(mask, axis=1)
    right = w - 1 - np.argmax(mask[:, ::-1], axis=1)
    width = np.where(any_row, right - left + 1, 0)

    rows   = np.flatnonzero(any_row)
    top_y  = int(ov.get("top_y")  or rows[0])
    foot_y = int(ov.get("foot_y") or rows[-1])
    if top_y <= 0:      warn.append("剪影贴到画面顶边，头顶可能被裁掉了")
    if foot_y >= h - 1: warn.append("剪影贴到画面底边，确认是全身图、脚底没被裁掉")

    total = foot_y - top_y + 1

    # 脖子 = 头顶下方 6%~55% 区间里的最窄行
    b0 = top_y + max(2, int(total*0.06))
    b1 = min(h - 1, top_y + int(total*0.55))
    if b1 <= b0:
        raise ValueError("图太小或剪影异常，量不出来")
    seg    = np.where(width[b0:b1+1] > 0, width[b0:b1+1], 10**9)
    neck_y = int(b0 + seg.argmin())
    neck_w = int(width[neck_y])
    head_top_max = int(width[top_y:neck_y+1].max())

    auto_chin = None
    if neck_w >= head_top_max*0.80:
        warn.append("找不到明显的颈部收窄（最窄 %d px 已是头宽 %d px 的 %.0f%%）——多半是长发"
                    "盖住了脖子。下巴线基本不可信，请手动给 chin_y。"
                    % (neck_w, head_top_max, 100.0*neck_w/max(1, head_top_max)))
    else:
        widest_y = int(top_y + width[top_y:neck_y+1].argmax())
        thr = neck_w*1.08
        auto_chin = neck_y
        for y in range(widest_y, neck_y+1):
            if width[y] <= thr:
                auto_chin = y; break

    chin_y = int(ov.get("chin_y") or (auto_chin if auto_chin is not None else neck_y))
    if chin_y <= top_y:
        raise ValueError("下巴线在头顶线之上，检查 chin_y")
    head_h = chin_y - top_y + 1

    hw_y   = int(top_y + width[top_y:chin_y+1].argmax())
    head_w = float(ov.get("head_w") or width[hw_y])

    s0 = neck_y + 1
    s1 = min(h - 1, neck_y + max(3, int(head_h*shoulder_band)))
    if s1 <= s0: s1 = min(h - 1, s0 + 3)
    sh_y       = int(s0 + width[s0:s1+1].argmax())
    shoulder_w = float(ov.get("shoulder_w") or width[sh_y])

    N = total/float(head_h)
    r = head_w/float(shoulder_w) if shoulder_w else float("nan")

    # 下巴线是整套方法最不可靠的一环，它一定落在 [下巴, 颈最窄] 之间。
    # 把这个区间对应的头身比范围报出来，让不确定度显式可见。
    N_alt = total/float(max(1, neck_y - top_y + 1))
    N_lo, N_hi = min(N, N_alt), max(N, N_alt)

    if not (1.8 <= N <= 9.5):
        warn.append("头身比 %.2f 超出常见范围 1.8~9.5，几乎肯定是分割错了" % N)
    if not (0.35 <= r <= 1.40):
        warn.append("头宽/肩宽 %.2f 超出常见范围 0.35~1.40，检查肩线" % r)
    if shoulder_w <= head_w:
        warn.append("肩比头还窄（肩 %.0f px ≤ 头 %.0f px）——长发把肩线盖住了，"
                    "量到的多半是头发不是肩膀，建议手动给 shoulder_w" % (shoulder_w, head_w))

    return dict(top_y=top_y, chin_y=chin_y, neck_y=neck_y, foot_y=foot_y,
                hw_y=hw_y, sh_y=sh_y, total=total, head_h=head_h,
                head_w=head_w, shoulder_w=shoulder_w, neck_w=neck_w,
                head_span=(int(left[hw_y]), int(right[hw_y])),
                shoulder_span=(int(left[sh_y]), int(right[sh_y])),
                N=N, r=r, N_lo=N_lo, N_hi=N_hi, width=width, warnings=warn,
                chin_src=("手动指定" if ov.get("chin_y") else
                          ("自动检出" if auto_chin is not None else "检不出，回退到颈最窄线")))


def plot_analysis(a, m, figsize=(11, 7)):
    h, w = a.shape[:2]
    fig, (ax, axp) = plt.subplots(1, 2, figsize=figsize, sharey=True,
                                  gridspec_kw={"width_ratios": [2.2, 1]})
    canvas = a.copy()
    if canvas[..., 3].min() < 255:                 # 透明处铺白，线才看得见
        al = canvas[..., 3:4]/255.0
        canvas = (canvas[..., :3]*al + 255*(1-al)).astype(np.uint8)
    else:
        canvas = canvas[..., :3]
    ax.imshow(canvas)

    for y, c, lab in ((m["top_y"], "#e83e3e", L("头顶", "top")),
                      (m["chin_y"], "#e83e3e", L("下巴", "chin")),
                      (m["foot_y"], "#3a84e8", L("脚底", "feet"))):
        ax.axhline(y, color=c, lw=1.4)
        ax.text(4, y-4, lab, color=c, fontsize=9, va="bottom", bbox=dict(fc="white", alpha=0.72, ec="none", pad=1), ha="left")
    ax.axhline(m["neck_y"], color="#999", lw=0.8, ls="--")

    hl, hr = m["head_span"]
    ax.annotate("", xy=(hl, m["hw_y"]), xytext=(hr, m["hw_y"]),
                arrowprops=dict(arrowstyle="<->", color="#2eb86a", lw=2))
    ax.text((hl+hr)/2, m["hw_y"]-10, L("头宽 %.0f", "head w %.0f") % m["head_w"],
            color="#2eb86a", fontsize=9, ha="center", bbox=dict(fc="white", alpha=0.72, ec="none", pad=1), va="bottom")
    sl, sr = m["shoulder_span"]
    ax.annotate("", xy=(sl, m["sh_y"]), xytext=(sr, m["sh_y"]),
                arrowprops=dict(arrowstyle="<->", color="#f09628", lw=2))
    ax.text((sl+sr)/2, m["sh_y"]+12, L("肩宽 %.0f", "shoulder w %.0f") % m["shoulder_w"],
            color="#f09628", fontsize=9, ha="center", bbox=dict(fc="white", alpha=0.72, ec="none", pad=1), va="top")
    ax.set_title(L("标注图 —— 四条线对不对，看一眼就知道",
                   "annotated: verify the four lines"), fontsize=10)
    ax.axis("off")

    axp.plot(m["width"], np.arange(h), color="#666", lw=1)
    for y, c in ((m["top_y"], "#e83e3e"), (m["chin_y"], "#e83e3e"),
                 (m["foot_y"], "#3a84e8"), (m["neck_y"], "#999")):
        axp.axhline(y, color=c, lw=0.8, ls="--")
    axp.set_title(L("逐行剪影宽度剖面", "silhouette width profile"), fontsize=10)
    # 不要在这里 invert_yaxis()：sharey 已经继承了 imshow 的倒置 y 轴，
    # 再翻一次会把整张图上下颠倒。
    axp.set_xlabel("px"); axp.grid(alpha=0.25)
    plt.tight_layout(); plt.show()

print("✅ 图像分析引擎就绪" + ("（中文字体可用）" if HAS_CJK else "（无中文字体，图上标签用英文）"))

## A · 上传立绘，自动量比例

带**透明背景的 PNG** 最准。没有立绘就跳过这一节，直接去 B 手填 `N` 和 `r`。

跑完对着标注图检查四条线；不对就在下面的 `chin_y` / `top_y` / `foot_y` / `head_w` / `shoulder_w`
填**原图像素坐标**覆盖（填 0 = 用自动值），重跑本 cell。

In [ ]:
#@title 【A】上传立绘并自动量比例 { display-mode: "form" }
#@markdown **图片来源**：留空则弹出上传框；也可以填 Colab 里的路径或图片 URL
image_source = "" #@param {type:"string"}
#@markdown ---
#@markdown **抠图参数**
工作分辨率上限 = 1000 #@param {type:"slider", min:400, max:2000, step:100}
背景色容差 = 30 #@param {type:"slider", min:5, max:90, step:5}
肩线搜索范围_乘头高 = 0.6 #@param {type:"slider", min:0.2, max:1.5, step:0.1}
#@markdown ---
#@markdown **手动覆盖**（原图像素坐标，0 = 用自动值）
chin_y = 0 #@param {type:"integer"}
top_y = 0 #@param {type:"integer"}
foot_y = 0 #@param {type:"integer"}
head_w = 0 #@param {type:"integer"}
shoulder_w = 0 #@param {type:"integer"}

src = image_source.strip()
if not src:
    from google.colab import files
    up = files.upload()
    if not up:
        raise SystemExit("没有选择文件")
    src = list(up.keys())[0]

a, scale = load_rgba(src, int(工作分辨率上限))
mask, how = build_mask(a, bg_tol=int(背景色容差))

# 覆盖值是原图坐标，换算到工作图坐标
ov = {k: (v/scale if v else None) for k, v in
      dict(top_y=top_y, chin_y=chin_y, foot_y=foot_y,
           head_w=head_w, shoulder_w=shoulder_w).items()}

m = analyse(mask, ov, float(肩线搜索范围_乘头高))
AUTO_N, AUTO_R = round(m["N"], 2), round(m["r"], 3)

print("图片 %s" % src)
print("抠图：%s    工作分辨率 %d×%d（原图约为其 %.2f 倍）"
      % (how, a.shape[1], a.shape[0], scale))
print("线位(工作图 y)：头顶 %d  下巴 %d  颈最窄 %d  脚底 %d   下巴线来源：%s"
      % (m["top_y"], m["chin_y"], m["neck_y"], m["foot_y"], m["chin_src"]))
print("全身高 %d px   头高 %d px   头宽 %.0f px   肩宽 %.0f px   颈宽 %d px"
      % (m["total"], m["head_h"], m["head_w"], m["shoulder_w"], m["neck_w"]))
print()
print("★ 头身比    N = %d / %d = %.2f" % (m["total"], m["head_h"], m["N"]))
if m["N_hi"] - m["N_lo"] > 0.01:
    print("            下巴线取在 [下巴, 颈最窄] 之间何处，会让 N 落在 %.2f ~ %.2f；"
          % (m["N_lo"], m["N_hi"]))
    print("            这是本方法的主要误差来源，对着标注图确认再用")
print("★ 头宽/肩宽 r = %.0f / %.0f = %.3f" % (m["head_w"], m["shoulder_w"], m["r"]))
print("  （头宽取剪影最大宽，天然含发，与 B 节 r 的定义一致）")
for s in m["warnings"]:
    print("⚠ " + s)

plot_analysis(a, m)
print("已存入 AUTO_N = %s，AUTO_R = %s，B 节可直接调用" % (AUTO_N, AUTO_R))

## B · 演员数据 → 头壳尺寸

`用A节自动值` 打勾就吃 A 的结果；不打勾就用你手填的 `头身比` / `头宽肩宽比`。

假发那几个值填 **0 表示按基准值自动派生**（系数：顶 1.0 / 侧 1.0 / 前 0.6 / 后 1.4）。
侧向的 1.0 只是没有实测时的兜底，能量到就直接填。

In [ ]:
#@title 【B】算头壳尺寸 { display-mode: "form" }
#@markdown ### 演员实测数据（cm / kg）
身高 = 168 #@param {type:"number"}
体重 = 52 #@param {type:"number"}
肩宽 = 38 #@param {type:"number"}
头宽 = 15.5 #@param {type:"number"}
头高 = 22.5 #@param {type:"number"}
#@markdown 下面两项留 0 = 按经验比例估算
头深 = 0 #@param {type:"number"}
肩峰高 = 0 #@param {type:"number"}

#@markdown ### 角色设定
用A节自动值 = True #@param {type:"boolean"}
头身比 = 6.0 #@param {type:"number"}
头宽肩宽比 = 0.65 #@param {type:"number"}
头深宽比 = 1.15 #@param {type:"slider", min:1.0, max:1.35, step:0.01}
头身比含发型 = True #@param {type:"boolean"}

#@markdown ### 假发留量（自头壳基底材质外表面量到发丝外缘，cm；0 = 自动派生）
基准留量 = 3.0 #@param {type:"number"}
顶 = 0 #@param {type:"number"}
侧_单侧 = 0 #@param {type:"number"}
前 = 0 #@param {type:"number"}
后 = 0 #@param {type:"number"}

#@markdown ### 海绵留量（贴壳内壁的衬垫**标称**厚度，cm；0 = 自动派生）
#@markdown 和下面的「空气余量」不是一回事：海绵是实体、可压缩、负责夹住头；空气余量是风扇/走线/通风的空隙
海绵基准厚度 = 2.0 #@param {type:"number"}
海绵_顶 = 0 #@param {type:"number"}
海绵_侧单侧 = 0 #@param {type:"number"}
海绵_面 = 0 #@param {type:"number"}
海绵_后 = 0 #@param {type:"number"}
海绵压缩率 = 0.3 #@param {type:"slider", min:0, max:0.8, step:0.05}
海绵密度_g_cm3 = 0.05 #@param {type:"number"}

#@markdown ### 制作参数
支撑方式 = "chin" #@param ["chin", "shoulder", "custom"]
底沿低于下巴 = 5.0 #@param {type:"number"}
增高鞋厚度 = 0 #@param {type:"number"}
壁厚 = 0.5 #@param {type:"number"}
单侧空气余量 = 1.0 #@param {type:"number"}
头顶空气余量 = 1.5 #@param {type:"number"}
面部空气余量 = 2.0 #@param {type:"number"}
后脑空气余量 = 1.5 #@param {type:"number"}
单侧肩部填充 = 0 #@param {type:"number"}
材料 = "pla" #@param ["pla", "petg", "abs", "resin", "glassfiber"]

from IPython.display import HTML, display
from html import escape as _esc   # 警告里有 "<" 之类的字符，不转义会被浏览器吞掉后半句

N = globals().get("AUTO_N") if 用A节自动值 else None
r = globals().get("AUTO_R") if 用A节自动值 else None
src_note = "取自 A 节自动测量" if (N and r) else "手填"
if not (N and r):
    N, r = 头身比, 头宽肩宽比

R = compute(
    height=身高, weight=体重, shoulder_width=肩宽, head_width=头宽, head_height=头高,
    ratio_N=N, head_shoulder_r=r,
    head_depth=(头深 or None), shoulder_height=(肩峰高 or None),
    depth_ratio=头深宽比, hair_in_ratio=头身比含发型,
    wig=基准留量, wig_top=(顶 or None), wig_side=(侧_单侧 or None),
    wig_front=(前 or None), wig_back=(后 or None),
    foam=海绵基准厚度, foam_top=(海绵_顶 or None), foam_side=(海绵_侧单侧 or None),
    foam_face=(海绵_面 or None), foam_back=(海绵_后 or None),
    foam_compress=海绵压缩率, foam_density=海绵密度_g_cm3,
    mount=支撑方式, rim_drop=底沿低于下巴, lift=增高鞋厚度,
    side_clear=单侧空气余量, top_clear=头顶空气余量,
    face_clear=面部空气余量, back_clear=后脑空气余量,
    wall=壁厚, shoulder_pad=(单侧肩部填充 or None), material=材料,
)
RESULT = R

def _rows(d, keys, unit=""):
    return "".join("<td style='text-align:right;padding:4px 14px'>%.1f%s</td>" % (d[k], unit)
                   for k in keys)


def _chain_text(R):
    """尺寸链：可戴最小净壳是怎么一段段堆出来的。"""
    names = {"W": "宽", "H": "高", "D": "深"}
    out = []
    for k in ("W", "H", "D"):
        expr = " + ".join("%s %.1f" % (nm, v) for nm, v in R["chain"][k])
        # 合计显示未取整的真值，避免和上表的最小值差 0.1
        out.append("%s：%s\n      = %.1f cm  ← 最小%s"
                   % (names[k], expr, R["minimum"][k], names[k]))
    return _esc("\n".join(out))

badge = ("<span style='background:#1f9d55;color:#fff;padding:3px 10px;border-radius:4px'>✓ 可行</span>"
         if R["ok"]["all"] else
         "<span style='background:#c53030;color:#fff;padding:3px 10px;border-radius:4px'>✗ 做不出来</span>")

html = """
<div style="font-family:-apple-system,'Segoe UI',sans-serif;line-height:1.55;max-width:900px">
<h3 style="margin:.2em 0">头壳尺寸 &nbsp; %s</h3>
<p style="color:#666;margin:.2em 0 1em">角色 %.2f 头身，头宽/肩宽 %.3f（%s）&nbsp;·&nbsp;
%s&nbsp;·&nbsp;底沿离地 %.1f cm&nbsp;·&nbsp;成品总身高 %.1f cm</p>

<table style="border-collapse:collapse;margin-bottom:1.2em">
<tr style="background:#f0f0f0"><th style="padding:6px 14px;text-align:left"></th>
<th style="padding:6px 14px">深(前后)</th><th style="padding:6px 14px">宽(左右)</th>
<th style="padding:6px 14px">高(上下)</th></tr>
<tr style="background:#fffbe6;font-weight:600"><td style="padding:4px 14px">净壳（建模用）</td>%s</tr>
<tr><td style="padding:4px 14px">含发外观（成品）</td>%s</tr>
<tr style="color:#888"><td style="padding:4px 14px">可戴最小值</td>%s</tr>
<tr style="color:#888"><td style="padding:4px 14px">余量</td>%s</tr>
</table>

<p><b>净壳比例</b> 深:宽:高 = %.3f : 1.000 : %.3f &nbsp;&nbsp;|&nbsp;&nbsp;
<b>建模包围盒</b> %.1f × %.1f × %.1f mm</p>

<h4 style="margin:1em 0 .3em">海绵留量与尺寸链</h4>
<p style="margin:.2em 0">标称厚度 顶 %.2f / 侧(单侧) %.2f / 面 %.2f / 后 %.2f cm&nbsp;&nbsp;—&nbsp;&nbsp;
压缩 %.0f%% → 实占 顶 %.2f / 侧 %.2f / 面 %.2f / 后 %.2f cm<br>
<span style="color:#888">海绵按标称厚度买，戴上后被压薄：占位算实占，重量算标称。</span></p>
<pre style="background:#f7f7f7;padding:10px 12px;border-radius:4px;font-size:12px;
line-height:1.7;overflow-x:auto;margin:.4em 0">%s</pre>

<p><b>眼位</b>：演员眼睛离壳顶 %.1f cm，占净壳高 %.1f%%（角色眼位建议 %.0f%%~%.0f%%）→ %s<br>
<b>重量</b>：%s 壁厚 %.1f mm —— 壳体 %.2f + 海绵 %.2f = <b>%.2f kg</b>
（上限 %.2f kg；涂装+假发+风扇另计 0.4~1.2 kg）<br>
<b>极限</b>：该演员最大可行 %.2f 头身；达成当前头宽需单侧肩部填充 %.1f cm%s</p>
</div>
""" % (
    badge, N, r, src_note, R["mount_desc"], R["h_mount"], R["total_height"],
    _rows(R["net"], "DWH"), _rows(R["vis"], "DWH"),
    _rows(R["minimum"], "DWH"),
    "".join("<td style='text-align:right;padding:4px 14px'>%+.1f</td>"
            % (R["net"][k]-R["minimum"][k]) for k in "DWH"),
    R["net"]["D"]/R["net"]["W"], R["net"]["H"]/R["net"]["W"],
    R["net"]["D"]*10, R["net"]["W"]*10, R["net"]["H"]*10,
    R["foam"]["nom_top"], R["foam"]["nom_side"], R["foam"]["nom_face"], R["foam"]["nom_back"],
    R["foam"]["compress"]*100,
    R["foam"]["eff_top"], R["foam"]["eff_side"], R["foam"]["eff_face"], R["foam"]["eff_back"],
    _chain_text(R),
    R["eye"]["from_top"], R["eye"]["ratio"]*100,
    EYE_FROM_TOP_MIN*100, EYE_FROM_TOP_MAX*100,
    "对得上" if R["eye"]["ok"] else "需内部调整头台高度",
    R["mass"]["material"], R["mass"]["wall_mm"], R["mass"]["shell_kg"],
    R["mass"]["foam_kg"], R["mass"]["total_kg"], R["mass"]["limit_kg"],
    R["n_max"], R["pad_needed"],
    ("；达成当前头身比还需增高 %.1f cm" % R["extra_lift"]) if R["extra_lift"] > 0 else "",
)
display(HTML(html))

for tag, items, color in (("估算说明", R["notes"], "#666"),
                          ("⚠ 警告", R["warnings"], "#b7791f"),
                          ("→ 建议", R["advice"], "#2b6cb0")):
    if items:
        display(HTML("<div style='font-family:-apple-system,sans-serif;color:%s;max-width:900px'>"
                     "<b>%s</b><ul style='margin:.3em 0'>%s</ul></div>"
                     % (color, tag, "".join("<li>%s</li>" % _esc(s) for s in items))))

## C · 可行性扫描

两张图回答两个问题：**这位演员最多能做到几头身？** **肩要垫多宽才够？**

阴影区 = 壳体小到塞不进人头的不可行区。

In [ ]:
#@title 【C】扫描图 { display-mode: "form" }
头身比扫描范围 = "3, 9" #@param {type:"string"}
头宽肩宽比扫描范围 = "0.4, 1.1" #@param {type:"string"}

import numpy as np, matplotlib.pyplot as plt
R = globals().get("RESULT")
if R is None:
    raise SystemExit("请先运行 B 节")

wt, ws_ = R["wig"]["top"], R["wig"]["side"]
h_mount, hs_min, ws_min = R["h_mount"], R["minimum"]["H"], R["minimum"]["W"]
hair = 头身比含发型

n0, n1 = [float(x) for x in 头身比扫描范围.split(",")]
r0, r1 = [float(x) for x in 头宽肩宽比扫描范围.split(",")]
Ns = np.linspace(max(1.05, n0), n1, 300)
Rs = np.linspace(r0, r1, 300)

hs = (h_mount/(Ns-1) - wt) if hair else ((h_mount + wt)/(Ns-1))
wsv = Rs*R["shoulder_cos"] - 2*ws_

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))

a1.plot(Ns, hs, color="#2b6cb0", lw=2)
a1.axhline(hs_min, color="#c53030", ls="--", lw=1.4)
a1.fill_between(Ns, 0, hs_min, color="#c53030", alpha=0.10)
a1.axvline(R["n_max"], color="#1f9d55", ls=":", lw=1.4)
a1.plot([globals().get("AUTO_N") or 头身比], [R["net"]["H"]], "o", color="#f09628", ms=9, zorder=5)
a1.set_xlabel(L("头身比 N", "head-body ratio N"))
a1.set_ylabel(L("净壳高 (cm)", "net shell height (cm)"))
a1.set_title(L("最大可行 %.2f 头身（绿虚线）", "max feasible N = %.2f") % R["n_max"], fontsize=10)
a1.set_ylim(0, max(hs_min*2.2, R["net"]["H"]*1.4)); a1.grid(alpha=0.25)
a1.text(Ns[-1], hs_min, L("  可戴下限 %.1f", "  min %.1f") % hs_min,
        color="#c53030", va="bottom", ha="right", fontsize=9)

a2.plot(Rs, wsv, color="#2b6cb0", lw=2)
a2.axhline(ws_min, color="#c53030", ls="--", lw=1.4)
a2.fill_between(Rs, 0, ws_min, color="#c53030", alpha=0.10)
a2.plot([globals().get("AUTO_R") or 头宽肩宽比], [R["net"]["W"]], "o", color="#f09628", ms=9, zorder=5)
a2.set_xlabel(L("头宽 / 肩宽  r", "head/shoulder ratio r"))
a2.set_ylabel(L("净壳宽 (cm)", "net shell width (cm)"))
a2.set_title(L("当前肩宽 %.1f cm 下", "at shoulder width %.1f cm") % R["shoulder_cos"], fontsize=10)
a2.set_ylim(0, max(ws_min*2.2, R["net"]["W"]*1.4)); a2.grid(alpha=0.25)
a2.text(Rs[-1], ws_min, L("  可戴下限 %.1f", "  min %.1f") % ws_min,
        color="#c53030", va="bottom", ha="right", fontsize=9)

plt.tight_layout(); plt.show()
print("橙点 = 当前设定。落在阴影区里就是做不出来。")